# 00 - Environment

Plan section 39: verify CUDA, GPUs, RAM, disk, PyTorch, and the declared V0 parameter contract before any session starts.

Reproducibility contract (plan section 40): every session begins by
determining its configuration, locating and validating prior state,
and continues from there; it ends by saving metrics and a summary JSON.
Run inside a Kaggle session (or locally) after checking out this
repository, with the repository root as the working directory.


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd()
assert (REPO / 'src' / 'act').is_dir(), (
    f'run this notebook with the repository root as the working directory; '
    f'cwd was {REPO!r}'
)
sys.path.insert(0, str(REPO / 'src'))
print('python', sys.version.split()[0], '| repo', REPO)


# Environment check (plan section 39)

Verifies CUDA visibility, GPU count, host RAM, disk, PyTorch, and -
the part the protocol actually depends on - that the declared V0
configuration materializes the pre-registered parameter counts
(total 285,336,576, active 134,341,632). The section 5.1 tolerance
band is derived from these counts, so a machine that cannot
reproduce them must not launch a session.


In [ ]:
import json, shutil, torch

env = {
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_version': torch.version.cuda,
    'gpu_count': torch.cuda.device_count() if torch.cuda.is_available() else 0,
    'gpus': [
        torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())
    ],
    'host_ram_gb': round(shutil.disk_usage('/').total / 2**30, 1),
    'free_disk_gb': round(shutil.disk_usage(os.getcwd()).free / 2**30, 1),
}
print(json.dumps(env, indent=2))

In [ ]:
from act.model import DECLARED_V0_CONFIG, estimated_parameter_count

total, active = estimated_parameter_count(DECLARED_V0_CONFIG)
assert total == 285_336_576, f'declared total parameters drifted: {total}'
assert active == 134_341_632, f'declared active parameters drifted: {active}'
print(f'declared V0 parameters: total={total:,} active={active:,}')

In [ ]:
from act.training.session_plan import (
    CHECKPOINT_PAYLOAD_GB,
    MAX_CHECKPOINTS,
    PairSchedule,
    decide_dtype,
    working_dir_headroom_gb,
)

# The declared schedule at the fp32 floor (section 5.2's registered row).
floor = PairSchedule(tokens_per_second=2_500.0)
print('fp32 floor pair time (h):', round(floor.pair_time_hours, 2))
print('full pairs per session:', floor.full_pairs_per_session)
print('primary sessions:', floor.sessions_for_primary,
      '| total incl. secondary:', floor.sessions_total)
print('checkpoint payload GB:', CHECKPOINT_PAYLOAD_GB,
      '| max_checkpoints pin:', MAX_CHECKPOINTS)
print('working-dir headroom GB:', round(working_dir_headroom_gb(), 2))
# The dtype rule is pre-committed; the session runner re-derives it
# from the benchmark artifact and refuses a stale one.
print('dtype at 2,500 tokens/s:', decide_dtype(2_500.0))
print('dtype below floor:', decide_dtype(2_499.0))

In [ ]:
import json
from pathlib import Path

record_dir = Path('artifacts/notebook-records')
record_dir.mkdir(parents=True, exist_ok=True)
payload = {'notebook': '00_environment', 'torch': env['torch'],
 'gpu_count': env['gpu_count'],
 'declared_total_parameters': total,
 'declared_active_parameters': active,
 'dtype_rule_floor_tokens_per_second': 2500.0}
record = json.dumps(payload, sort_keys=True, indent=2) + '\n'
target = record_dir / '00-environment.json'
existing = target.read_text() if target.exists() else None
if existing != record:
    target.write_text(record)
print('record:', target)